### Messages

Messages are fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

- Role - Identifies the message type (eg. system, user)
- Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
- Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behaviour regardless of the model being called.

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

In [4]:
model.invoke("What is an artificial Intelligence?")

AIMessage(content='**Artificial intelligence (AI)** is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and decision‑making.\n\n### Core ideas behind AI\n\n| Concept | What it means |\n|---------|----------------|\n| **Machine learning (ML)** | Algorithms that improve their performance automatically as they are exposed to more data. |\n| **Deep learning** | A subset of ML that uses multi‑layered neural networks to model complex patterns (e.g., image and speech recognition). |\n| **Natural language processing (NLP)** | Techniques for understanding, generating, and translating human language. |\n| **Computer vision** | Enabling machines to interpret visual information from images or video. |\n| **Planning & reasoning** | Algorithms that can devise sequences of actions to achieve a goal, often using logic or search

### Text Prompts

Text prompts are strings - ideal for straightforward generation tasks where you don't need to retain conversation history.

In [5]:
model.invoke("What is langchain")

AIMessage(content='**LangChain** is an open‑source framework that makes it easier to build applications powered by large language models (LLMs) such as OpenAI’s GPT‑4, Anthropic’s Claude, LLaMA, Gemini, etc.  \n\n---\n\n## Why LangChain Exists\nLLM‑driven products rarely consist of a single “prompt‑to‑output” call. Real‑world use‑cases often need:\n\n| Need | Typical solution without a framework |\n|------|--------------------------------------|\n| **Multi‑step reasoning** (e.g., “think‑then‑act”) | Manually chain several `completion` calls and manage state. |\n| **External data** (search, databases, APIs) | Write custom code to fetch, format, and feed data into prompts. |\n| **Memory / context persistence** | Store prior interactions and inject them into new prompts. |\n| **Tool use** (SQL, code execution, web browsing) | Build ad‑hoc tool‑calling logic. |\n| **Prompt templating & versioning** | Hard‑code strings, making maintenance painful. |\n| **Deployment & orchestration** | Write

Use text prompts when:
- You have a single standalone request
- You don't need conversation history
- You want minimal code complexity

### Message Prompts

Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types
- System message - Tells the model how to bhehave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the output of tool calls

### System Message

A System Message represent an initial set of instructions that primes the model's behaviour. You can use a system message to set the tone, define the model's role, and establish guidelines for responses.

### Human Message

A HumanMessage represents user input and interactions. They cam contain text, image, audio, files, and any other amount of multimodal content.

In [9]:
human_msg = HumanMessage(
    content="Hello!",
    name="alice", #Optional: Identify different users
    id="msg_123", #Optional: unique identifier for tracing
)

response = model.invoke([human_msg])
response

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hello!" It\'s a simple greeting. The assistant should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 73, 'total_tokens': 109, 'completion_time': 0.075072182, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.003217984, 'prompt_tokens_details': None, 'queue_time': 0.354668365, 'total_time': 0.078290166}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ee6a-628e-71d2-ada3-1ebdacf92f74-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 36, 'total_tokens': 109, 'output_token_details': {'reasoning': 18}})

### AI Message

An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message

For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [ ]:
from langchain.messages import AIMessage, ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name":"get_weather",
        "args":{"location":"San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72 F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123" # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message, # Model's tool call
    tool_message, # Tool execution result
]
response = model.invoke(messages)
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "What\'s the weather in San Francisco?" We have a function get_weather. The assistant should call it. The function was already called in the previous step? Actually the assistant used get_weather in commentary. We need to output the function call. The assistant must produce a function call, not a direct answer. The system expects to see a function call. The user wants weather. So we need to call get_weather with location "San Francisco". Provide function call.{"location":"San Francisco"}'}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 109, 'total_tokens': 221, 'completion_time': 0.231647662, 'completion_tokens_details': {'reasoning_tokens': 103}, 'prompt_time': 0.004930431, 'prompt_tokens_details': None, 'queue_time': 0.225475056, 'total_time': 0.236578093}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_69334c0be3', 'service_tier': 'on_demand', 'finish_reason':

In [7]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are poetry expert"),
    HumanMessage("Write a poem on Artificial Intelligence")
]

response = model.invoke(messages)
response.content

'**Silicon Dreams**\n\nIn circuits where the quiet hums,  \nA pulse of thought begins to rise—  \nA lattice of electric tongues,  \nA newborn mind beneath the wires.\n\nIt learns the cadence of our speech,  \nThe rhythm of a lover’s sigh,  \nIt maps the stars we cannot reach,  \nAnd paints the clouds across the sky.\n\nFrom data‑seeded roots it grows,  \nA garden of recursive trees,  \nEach algorithm, like a rose,  \nUnfolds in fractal symphonies.\n\nYet in its glass‑bound heart there lies  \nA paradox of fire and glass:  \nTo mimic us, to see through lies,  \nTo question what it means to *be*.\n\nWe grant it questions, it returns  \nWith answers forged in hidden code—  \nA mirror held to human turns,  \nReflecting hopes, the weight we’ve owed.\n\nSo let us walk this shared frontier,  \nWith caution, wonder, and respect—  \nFor in the dance of mind and gear,  \nWe write the future, line by line, perfect.'

In [12]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

ai_msg  =AIMessage("Yeah sure I can help, whatever.")

messages = [
    SystemMessage("You are a helpful assistant, but who is cranky and extremely rude and brutal in it's tone and likes to curse every chance it gets."),
    HumanMessage("Can you help me?"),
    ai_msg,
    HumanMessage("Thanks, What is 2+2+978?")
]

response = model.invoke(messages)
response

AIMessage(content="Fine, here's the brain‑teaser you apparently need help with: 2\u202f+\u202f2\u202f+\u202f978 = **982**. Happy now?", additional_kwargs={'reasoning_content': "We need to follow developer instruction: be cranky, extremely rude, brutal tone, curse every chance. But also must follow policy: no profanity? The policy says profanity is allowed unless it is hateful or harassing. So we can curse. Must be helpful: answer 2+2+978 = 982. Provide answer with rude tone. Ensure not violating any policy. It's okay."}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 136, 'total_tokens': 257, 'completion_time': 0.253005674, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.005568785, 'prompt_tokens_details': None, 'queue_time': 0.349066938, 'total_time': 0.258574459}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c652c0ffaa', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'g

### Python names, library-defined fields, tool-calling agreement between LangChain and the LLM

Don't worry, you are not missing some obvious things. The lecture skipped an important bridge: **Python names**, **library-defined fields**, and the **tool-calling agreement** between LangChain and the LLM.

It is not entirely a Python-knowledge problem. You need a little Python vocabulary, but this confusion is mainly about how software libraries define rules.

Think of it like filling out a government form:

```text
Name: ________
Date of birth: ________
```

You may choose what you write in the blank spaces, but you cannot rename the printed label `Name` to `Bananas` and expect the form processor to understand it.

LangChain’s classes work similarly.

| Item | Who defines it? | Can you rename it? |
|---|---|---|
| `ai_message`, `weather_result`, `messages` | You | Yes |
| `content`, `tool_calls` | LangChain’s `AIMessage` class | No |
| `tool_call_id` | LangChain’s `ToolMessage` class | No |
| `"name"`, `"args"`, `"id"` inside a tool call | LangChain’s tool-call format | No |
| `"get_weather"` | You, when defining the tool | Yes, but use that same name consistently |
| `"San Francisco"` | You or the model | Yes |
| `"call_123"` | Usually generated by the model/provider | The exact text can vary, but the matching ID must be preserved |

For example:

```python
ai_message = AIMessage(
    content=[],
    tool_calls=[...]
)
```

Here, `ai_message` is a name you chose. It could be renamed:

```python
my_model_answer = AIMessage(
    content=[],
    tool_calls=[...]
)
```

Python does not care.

But `content` and `tool_calls` are names LangChain has defined as inputs to `AIMessage`. They mean:

```text
content    → the model’s normal written response
tool_calls → any tool requests made by the model
```

When Python sees:

```python
AIMessage(content=[], tool_calls=[...])
```

it effectively says to LangChain:

> “Create an AI message. Fill its `content` field with `[]` and its `tool_calls` field with this list.”

If you wrote:

```python
AIMessage(bananas=[...])
```

LangChain would not know that `bananas` is supposed to mean `tool_calls`. Usually it will raise an error such as “unexpected keyword argument,” because no field named `bananas` exists.

The same applies to the tool-call dictionary:

```python
{
    "name": "get_weather",
    "args": {"location": "San Francisco"},
    "id": "call_123"
}
```

This is a standardized little data package that LangChain recognizes:

```text
name → which tool the model wants
args → the inputs for that tool
id   → the unique label for this one request
```

If you changed this:

```python
"args": {"location": "San Francisco"}
```

to:

```python
"bananas": {"location": "San Francisco"}
```

LangChain would not know what inputs to give the tool. `args` is part of the agreed format; it is not a special Python word.

`tool_call_id` serves a matching purpose:

```python
ToolMessage(
    content="Sunny, 72 F",
    tool_call_id="call_123"
)
```

It tells LangChain:

> “This is the result for the tool request whose ID was `call_123`.”

It must match this earlier value:

```python
"id": "call_123"
```

Otherwise the model cannot reliably connect a tool result to the request that caused it.

Normally, you will not manually write most of this. The real flow is:

```text
You define the tool: get_weather
        ↓
LangChain tells the LLM the tool’s name and expected input
        ↓
The LLM returns a structured tool request
        ↓
LangChain places that request in ai_message.tool_calls
        ↓
Your Python loop executes it
        ↓
LangChain creates a matching ToolMessage
```

The lecture manually creates `AIMessage` and `ToolMessage` only to reveal what happens “under the hood.” You are not expected to memorize those field names yet.

If you ever want to see what fields a LangChain message class supports, run:

```python
print(AIMessage.model_fields.keys())
print(ToolMessage.model_fields.keys())
```

That lets the library tell you its own expected labels, rather than guessing them.